# NLP Copilot Walkthrough
Notebook ini menjelaskan proses NLP yang dipakai di CS Copilot: TF-IDF + Logistic Regression untuk intent, TF-IDF similarity untuk FAQ, dan regex untuk entity extraction.

In [1]:
# Import libraries
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [2]:
# Konfigurasi path output
from pathlib import Path
import joblib
import json

BASE_DIR   = Path(".").resolve()
OUTPUT_DIR = (BASE_DIR.parent / "chatbot_models" / "logistic_regression").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH      = OUTPUT_DIR / "intent_model.pkl"
VECTORIZER_PATH = OUTPUT_DIR / "intent_vectorizer.pkl"
LABEL_MAP_PATH  = OUTPUT_DIR / "intent_label_map.json"

print("Output dir:", OUTPUT_DIR)

Output dir: C:\Users\Fahrezi\Documents\KULIAH\churn-dashboard\backend\app\chatbot_models\logistic_regression


## 1) Data intent (paraphrase bank)
Kumpulan contoh kalimat untuk melatih intent classifier.

In [3]:
# Load data latih dari CSV yang sudah digenerate
import pandas as pd
from pathlib import Path

DATASET_PATH = Path(".").resolve() / "intent_dataset.csv"
df_train     = pd.read_csv(DATASET_PATH)

TRAINING_SAMPLES = list(zip(df_train["text"].tolist(), df_train["intent"].tolist()))
texts  = df_train["text"].tolist()
labels = df_train["intent"].tolist()

print(f"Dataset     : {DATASET_PATH}")
print(f"Total sampel: {len(TRAINING_SAMPLES)}")
intents = sorted(df_train["intent"].unique().tolist())
print(f"Intent unik : {intents}")
print()
print("Per-intent count:")
print(df_train["intent"].value_counts().sort_index().to_string())
set(labels)


Dataset     : C:\Users\Fahrezi\Documents\KULIAH\churn-dashboard\backend\app\nlp_raw_model\intent_dataset.csv
Total sampel: 55000
Intent unik : ['ANALISIS_PELANGGAN', 'DRAF_EMAIL', 'FAKTOR_CHURN', 'GREETING', 'JUMLAH_RISIKO_TINGGI', 'METRIK_OVERVIEW', 'MODEL_INFO', 'SEGMEN_ANALISIS', 'STRATEGI_RETENSI', 'TREN_CHURN', 'VIP_RISK']

Per-intent count:
intent
ANALISIS_PELANGGAN      5000
DRAF_EMAIL              5000
FAKTOR_CHURN            5000
GREETING                5000
JUMLAH_RISIKO_TINGGI    5000
METRIK_OVERVIEW         5000
MODEL_INFO              5000
SEGMEN_ANALISIS         5000
STRATEGI_RETENSI        5000
TREN_CHURN              5000
VIP_RISK                5000


{'ANALISIS_PELANGGAN',
 'DRAF_EMAIL',
 'FAKTOR_CHURN',
 'GREETING',
 'JUMLAH_RISIKO_TINGGI',
 'METRIK_OVERVIEW',
 'MODEL_INFO',
 'SEGMEN_ANALISIS',
 'STRATEGI_RETENSI',
 'TREN_CHURN',
 'VIP_RISK'}

## 2) Train intent classifier
Gunakan TF-IDF (ngram 1-2) + Logistic Regression.

In [4]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
X = vectorizer.fit_transform(texts)
clf = LogisticRegression(max_iter=1000)
clf.fit(X, labels)
clf.classes_

array(['ANALISIS_PELANGGAN', 'DRAF_EMAIL', 'FAKTOR_CHURN', 'GREETING',
       'JUMLAH_RISIKO_TINGGI', 'METRIK_OVERVIEW', 'MODEL_INFO',
       'SEGMEN_ANALISIS', 'STRATEGI_RETENSI', 'TREN_CHURN', 'VIP_RISK'],
      dtype='<U20')

## 3) Predict intent + confidence
Lihat prediksi intent dan confidence score.

In [5]:
def predict_intent(text):
    Xq = vectorizer.transform([text])
    probs = clf.predict_proba(Xq)[0]
    idx = int(np.argmax(probs))
    return clf.classes_[idx], float(probs[idx])

tests = [
    # Cek DRAF_EMAIL
    "Bikin email retention buat C-0001",
    "Tolong susun email diskon untuk C-0008",
    
    # Cek FAKTOR_CHURN
    "Faktor churn dominan minggu ini apa?",
    "Tunjukkan faktor penting churn",
    
    # Cek ANALISIS_PELANGGAN
    "Ringkasan C-0010",
    "Profil customer C-0021",
    
    # Cek GREETING
    "Bisa bantu apa?",
    "Aku bisa tanya apa saja?",
    
    # Cek MODEL_INFO
    "Model AI apa yang dipakai chatbot ini?",
    
    # Cek METRIK_OVERVIEW
    "Tolong berikan ringkasan dashboard bulan ini",
    
    # Cek STRATEGI_RETENSI
    "Saran untuk mengurangi churn rate dong"
]
for t in tests:
    print(t, '->', predict_intent(t))

Bikin email retention buat C-0001 -> (np.str_('DRAF_EMAIL'), 0.6135532230454503)
Tolong susun email diskon untuk C-0008 -> (np.str_('DRAF_EMAIL'), 0.7882833382633883)
Faktor churn dominan minggu ini apa? -> (np.str_('FAKTOR_CHURN'), 0.6721581089320708)
Tunjukkan faktor penting churn -> (np.str_('FAKTOR_CHURN'), 0.6540652601787713)
Ringkasan C-0010 -> (np.str_('METRIK_OVERVIEW'), 0.4265030752735306)
Profil customer C-0021 -> (np.str_('ANALISIS_PELANGGAN'), 0.8877752229552938)
Bisa bantu apa? -> (np.str_('GREETING'), 0.9461553313961906)
Aku bisa tanya apa saja? -> (np.str_('GREETING'), 0.9836758394314584)
Model AI apa yang dipakai chatbot ini? -> (np.str_('MODEL_INFO'), 0.9946728884811847)
Tolong berikan ringkasan dashboard bulan ini -> (np.str_('METRIK_OVERVIEW'), 0.9869811059333423)
Saran untuk mengurangi churn rate dong -> (np.str_('STRATEGI_RETENSI'), 0.543638508452436)


## 4) FAQ retrieval (TF-IDF similarity)
Jika intent tidak jelas, cari jawaban FAQ paling mirip.

In [6]:
FAQ_ITEMS = [
    ("Cara bikin email penawaran diskon?", "Ketik: 'Tolong buatkan draf email penawaran diskon untuk pelanggan C-0001'."),
    ("Cara melihat faktor utama churn?", "Ketik: 'Apa faktor utama churn bulan ini?'."),
    ("Cara melihat ringkasan pelanggan?", "Ketik: 'Ringkasan pelanggan C-0001'."),
    ("Apa saja fitur copilot?", "Copilot bisa membuat draf email, merangkum faktor churn, dan memberi ringkasan pelanggan."),
]
faq_vectorizer = TfidfVectorizer(ngram_range=(1, 2))
faq_matrix = faq_vectorizer.fit_transform([q for q, _ in FAQ_ITEMS])

def retrieve_faq(query):
    q = faq_vectorizer.transform([query])
    scores = (faq_matrix @ q.T).toarray().ravel()
    best_idx = int(np.argmax(scores))
    return FAQ_ITEMS[best_idx], float(scores[best_idx])

retrieve_faq('gimana cara lihat faktor churn?')

(('Cara melihat faktor utama churn?',
  "Ketik: 'Apa faktor utama churn bulan ini?'."),
 0.5609542413727663)

## 5) Entity extraction (customer_id)
Ambil customer_id dengan regex.

In [7]:
def extract_customer_id(text):
    match = re.search(r'c-\d+', text, re.IGNORECASE)
    return match.group(0) if match else ''

extract_customer_id('Ringkasan pelanggan C-0001 dong')

'C-0001'

In [8]:
# Simpan Model, Vectorizer, dan Label Map ke disk
unique_labels = sorted(list(set(labels)))
label_to_idx  = {lbl: i for i, lbl in enumerate(unique_labels)}
idx_to_label  = {i: lbl for lbl, i in label_to_idx.items()}

# Simpan model classifier (.pkl)
joblib.dump(clf, MODEL_PATH)
print(f"[SIMPAN] Model      -> {MODEL_PATH}")

# Simpan vectorizer TF-IDF (.pkl)
joblib.dump(vectorizer, VECTORIZER_PATH)
print(f"[SIMPAN] Vectorizer -> {VECTORIZER_PATH}")

# Simpan label map (.json) — format sama persis dengan Deep Learning
with open(LABEL_MAP_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "label_to_idx": label_to_idx,
        "idx_to_label": {str(k): v for k, v in idx_to_label.items()},
    }, f, ensure_ascii=False, indent=2)
print(f"[SIMPAN] Label map  -> {LABEL_MAP_PATH}")


[SIMPAN] Model      -> C:\Users\Fahrezi\Documents\KULIAH\churn-dashboard\backend\app\chatbot_models\logistic_regression\intent_model.pkl
[SIMPAN] Vectorizer -> C:\Users\Fahrezi\Documents\KULIAH\churn-dashboard\backend\app\chatbot_models\logistic_regression\intent_vectorizer.pkl
[SIMPAN] Label map  -> C:\Users\Fahrezi\Documents\KULIAH\churn-dashboard\backend\app\chatbot_models\logistic_regression\intent_label_map.json


## Ringkas alur NLP
1. Deteksi intent dengan TF-IDF + Logistic Regression
2. Jika confidence rendah, gunakan FAQ retrieval
3. Extract customer_id dengan regex
4. Panggil action yang sesuai (email draft, churn factors, summary customer)